In [0]:
import glob
import shutil
import os

src_dir = '/Workspace/Users/scott.drummond@hhchealth.org/demo/seeds'
dst_dir = '/Volumes/learn_tuva/landing/raw_files'

files = glob.glob(f'{src_dir}/*.csv')
for f in files:
    fname = os.path.basename(f)
    shutil.copy2(f, os.path.join(dst_dir, fname))
    print(f"Copied: {fname}")

print(f"\nDone — {len(files)} file(s) landed to {dst_dir}")


In [0]:
catalog = 'learn_tuva'
schema = 'input_layer'

affected_tables = [
    'appointment', 'eligibility', 'immunization', 'lab_result',
    'medical_claim', 'observation', 'pharmacy_claim', 'provider_attribution_source'
]

for tname in affected_tables:
    table_ref = f"{catalog}.{schema}.{tname}"
    history = spark.sql(f"DESCRIBE HISTORY {table_ref}").orderBy("version").collect()
    max_version = max(v.version for v in history)

    if max_version == 0:
        # Table was created fresh by the agent — drop it to fully revert
        spark.sql(f"DROP TABLE IF EXISTS {table_ref}")
        print(f"{tname}: dropped (was created fresh, no prior version)")
    else:
        restore_to = max_version - 1
        spark.sql(f"RESTORE TABLE {table_ref} TO VERSION AS OF {restore_to}")
        count = spark.table(table_ref).count()
        print(f"{tname}: restored to v{restore_to} ({count} rows)")

print("\nRestore complete.")


In [0]:
catalog = 'learn_tuva'
schema = 'input_layer'
dst_dir = '/Volumes/learn_tuva/landing/raw_files'

tables = spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()

for row in tables:
    tname = row.tableName
    if tname.startswith('input_layer__'):
        print(f"Skipped (excluded): {tname}")
        continue

    table_ref = f"{catalog}.{schema}.{tname}"
    df = spark.table(table_ref)
    out_path = f"{dst_dir}/{tname}.csv"

    df.toPandas().to_csv(out_path, index=False)
    print(f"Exported: {table_ref} → {tname}.csv  ({df.count()} rows)")

print("\nDone.")
